# CogVideoX-5B — Image-to-Video Flying Animation

Generate a high-quality flying video from a single image using **CogVideoX-5B** (5B params, Tsinghua/ZhipuAI).

- **Model**: CogVideoX-5B-I2V (image-to-video) — the largest open video model that fits on Colab
- **Output**: 49 frames at 720×480, ~6 seconds at 8fps
- **VRAM**: ~18GB (float16) or ~12GB (INT8 quantized)
- **Runtime**: **A100 recommended** (40GB). T4 works with INT8 but is slow (~10 min/video).

Go to **Runtime > Change runtime type > A100 GPU** (or T4 if on free tier)

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors pillow imageio[ffmpeg] bitsandbytes sentencepiece

import torch
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory
    print(f'GPU: {props.name} — {vram / 1024**3:.1f} GiB')
    USE_QUANTIZATION = vram < 30 * 1024**3
    if USE_QUANTIZATION:
        print('Will use INT8 quantization (GPU has < 30GB VRAM)')
    else:
        print('Enough VRAM for float16 — no quantization needed')
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > A100 or T4')
print('Dependencies installed.')

In [ ]:
# Cell 2: Load CogVideoX-5B image-to-video pipeline
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
import torch

MODEL_ID = 'THUDM/CogVideoX-5b-I2V'

if USE_QUANTIZATION:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
    )
    pipe = CogVideoXImageToVideoPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        quantization_config=quantization_config,
    )
else:
    pipe = CogVideoXImageToVideoPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
    )

pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print(f'CogVideoX-5B I2V loaded! Quantized: {USE_QUANTIZATION}')

In [ ]:
# Cell 3: Load character reference image
import urllib.request
from PIL import Image

IMG_URL = 'https://raw.githubusercontent.com/mangeshgwagle/attestor/main/training/character_ref.webp'
IMG_PATH = '/content/character_ref.webp'

print('Downloading character image...')
urllib.request.urlretrieve(IMG_URL, IMG_PATH)
ref_image = Image.open(IMG_PATH).convert('RGB').resize((720, 480))
print(f'Loaded: {ref_image.size}')
display(ref_image)

In [ ]:
# Cell 4: Generate flying animation
import torch

PROMPT = (
    'An angelic warrior character with golden armor and white wings '
    'takes flight, soaring upward through dramatic clouds. '
    'The wings spread wide as the character rises into a glowing celestial sky. '
    'Wind flows through their cape. Epic fantasy cinematic, high quality.'
)

generator = torch.manual_seed(42)

video_frames = pipe(
    image=ref_image,
    prompt=PROMPT,
    num_frames=49,
    guidance_scale=6.0,
    num_inference_steps=50,
    generator=generator,
).frames[0]

print(f'Generated {len(video_frames)} frames!')

In [ ]:
# Cell 5: Export to MP4 and display inline
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode
import os

MP4_PATH = '/content/cogvideo_flying.mp4'
export_to_video(video_frames, MP4_PATH, fps=8)

mp4_size = os.path.getsize(MP4_PATH) / 1024**2
print(f'Video: {mp4_size:.1f} MB, {len(video_frames)} frames at 8fps = {len(video_frames)/8:.1f}s')

with open(MP4_PATH, 'rb') as f:
    mp4 = b64encode(f.read()).decode()
display(HTML(f'''
<video width="720" controls autoplay loop>
  <source src="data:video/mp4;base64,{mp4}" type="video/mp4">
</video>
'''))
print('Video is playing above!')

In [ ]:
# Cell 6: Generate variant — battle flight
import torch

PROMPT_V2 = (
    'An angelic warrior character diving through storm clouds at high speed, '
    'golden armor glowing with energy, wings folded for a dive, '
    'lightning flashing around them, dramatic action scene, '
    'fantasy game cinematic, motion blur, high quality.'
)

generator = torch.manual_seed(123)

video_frames_v2 = pipe(
    image=ref_image,
    prompt=PROMPT_V2,
    num_frames=49,
    guidance_scale=6.0,
    num_inference_steps=50,
    generator=generator,
).frames[0]

V2_PATH = '/content/cogvideo_flying_v2.mp4'
export_to_video(video_frames_v2, V2_PATH, fps=8)
print(f'Variant 2: {os.path.getsize(V2_PATH)/1024**2:.1f} MB')

In [ ]:
# Cell 7: Download all videos
from google.colab import files
import glob, os

for f in sorted(glob.glob('/content/cogvideo_*.mp4')):
    size = os.path.getsize(f) / 1024**2
    print(f'Downloading {os.path.basename(f)} ({size:.1f} MB)...')
    files.download(f)

print('Done! Check the Files sidebar (folder icon) if downloads did not trigger.')